In [ ]:
import re
from pathlib import Path
import pandas as pd
from rapidfuzz import process, fuzz
from unidecode import unidecode
from tqdm import tqdm

In [ ]:
CSMAR_PATH = Path("./stata_output/CN_CN/CSMAR_Engname.csv")
WOS_PATH   = Path("./stata_output/CN_CN/WOS_Engname.csv")

In [ ]:
CSMAR_ID_COL   = "Symbol"
CSMAR_NAME_COL = "English_name"
WOS_ID_COL     = "firm_id_wos"
WOS_NAME_COL   = "English_name_wos"

# === Tunable parameters ===
TOP_N = 3               # Top candidates kept per query
SCORE_CUTOFF = 95       # 80-92; higher is stricter
ENFORCE_ONE_TO_ONE = True  # Whether to enforce one-to-one greedy dedup on best matches

LEGAL_SUFFIXES = r"""
co|co\.|company|corp|corporation|inc|inc\.|ltd|ltd\.|limited|plc|llc|lp|llp|
holdings?|group|intl|international|the
"""

def normalize_name(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = unidecode(s).lower()
    s = re.sub(r"[&@/.,\-–—'’\"(){}\[\]|+*!?:;^~`•·]", " ", s)   # Remove punctuation/symbols
    s = re.sub(rf"\b(?:{LEGAL_SUFFIXES})\b", " ", s)              # Remove legal suffixes
    s = re.sub(r"\s+", " ", s).strip()
    return s

def first_token(s: str) -> str: return (s.split() + [""])[0]
def last_token(s: str)  -> str: return (s.split() + [""])[-1]
def prefix3(s: str)     -> str: return s[:3]

def main():
    # Load data
    a = pd.read_csv(CSMAR_PATH)
    b = pd.read_csv(WOS_PATH)

    # Basic checks
    for df, cols, name in [(a,[CSMAR_ID_COL,CSMAR_NAME_COL],"CSMAR"),
                           (b,[WOS_ID_COL,WOS_NAME_COL],"WOS")]:
        missing = [c for c in cols if c not in df.columns]
        if missing:
            raise ValueError(f"{name} missing columns: {missing}. Actual columns: {list(df.columns)}")

    # Normalize
    a["_raw"]  = a[CSMAR_NAME_COL].astype(str)
    b["_raw"]  = b[WOS_NAME_COL].astype(str)
    a["_norm"] = a["_raw"].map(normalize_name)
    b["_norm"] = b["_raw"].map(normalize_name)

    # Blocking keys
    for df in (a, b):
        df["_ftok"] = df["_norm"].map(first_token)
        df["_ltok"] = df["_norm"].map(last_token)
        df["_p3"]   = df["_norm"].map(prefix3)

    # Candidate indices
    idx_ftok = b.groupby("_ftok").indices
    idx_ltok = b.groupby("_ltok").indices
    idx_p3   = b.groupby("_p3").indices

    def candidate_indices(row):
        c = set()
        for key, mp in ((row["_ftok"], idx_ftok), (row["_ltok"], idx_ltok), (row["_p3"], idx_p3)):
            if key in mp:
                c.update(mp[key])
        return list(sorted(c)) if c else list(range(len(b)))  # Fallback to full table when no candidates

    # Match (with progress bar)
    rows = []
    for i, r in tqdm(a.iterrows(), total=len(a), desc="Matching"):
        q = r["_norm"]
        if not q:
            rows.append({ "cs_idx": i, "wos_idx": None, "score": 0, "strategy": "empty_name" })
            continue

        cand_idx   = candidate_indices(r)
        cand_names = b.iloc[cand_idx]["_norm"].tolist()

        matches = process.extract(q, cand_names, scorer=fuzz.token_set_ratio,
                                  score_cutoff=SCORE_CUTOFF, limit=TOP_N)
        strategy = "blocked_strict"
        if not matches:
            matches = process.extract(q, cand_names, scorer=fuzz.token_set_ratio,
                                      score_cutoff=max(70, SCORE_CUTOFF-10), limit=TOP_N)
            strategy = "blocked_loose" if matches else "no_match"

        if matches:
            for _, m_score, m_pos in matches:
                j = int(cand_idx[m_pos])   # Explicit cast to int
                rows.append({ "cs_idx": i, "wos_idx": j, "score": int(m_score), "strategy": strategy })
        else:
            rows.append({ "cs_idx": i, "wos_idx": None, "score": 0, "strategy": strategy })

    cand = pd.DataFrame(rows)

    # Expand ids/names - assign via mask in one shot to avoid iloc dtype issues
    cand["Symbol"]       = a.loc[cand["cs_idx"], CSMAR_ID_COL].values
    cand["English_name"] = a.loc[cand["cs_idx"], CSMAR_NAME_COL].values

    cand["firm_id_wos"] = None
    cand["English_name_wos"] = None
    mask = cand["wos_idx"].notna()
    if mask.any():
        idx = cand.loc[mask, "wos_idx"].astype(int)
        cand.loc[mask, "firm_id_wos"]      = b.iloc[idx][WOS_ID_COL].values
        cand.loc[mask, "English_name_wos"] = b.iloc[idx][WOS_NAME_COL].values

    # best: keep highest score per CSMAR; use length difference as tiebreaker
    def pick_best(g):
        g = g.copy()
        len_diff = (g["English_name_wos"].fillna("").str.len() - g["English_name"].fillna("").str.len()).abs()
        g["len_diff"] = len_diff
        return g.sort_values(["score","len_diff"], ascending=[False,True]).head(1)

    best = cand.groupby("cs_idx", as_index=False, group_keys=False).apply(pick_best)
    best = best.drop(columns=["len_diff"])

    # Optional one-to-one greedy: sort by score desc; each WOS matched once
    if ENFORCE_ONE_TO_ONE:
        best = best.sort_values("score", ascending=False)
        best = best.drop_duplicates(subset=["firm_id_wos"], keep="first").sort_values("cs_idx")

    # Export
    best_out = best[["Symbol","English_name","firm_id_wos","English_name_wos","score","strategy"]].copy()
    cand_out = cand[["Symbol","English_name","firm_id_wos","English_name_wos","score","strategy"]].copy()

    best_path = "./proc_output/WOS/name_match_best.csv"
    cand_path = "./proc_output/WOS/name_match_candidates_topN.csv"
    best_out.to_csv(best_path, index=False)
    cand_out.to_csv(cand_path, index=False)

    print("Saved:")
    print(f" - {best_path}  (Best match for each CSMAR name; one-to-one={ENFORCE_ONE_TO_ONE})")
    print(f" - {cand_path}  (Top-{TOP_N} candidate details)")

if __name__ == "__main__":
    main()